# Pet Shop Product Recommendations

> **Sample data disclaimer:** This notebook uses sample data, not the original business data. Recommendation examples and association metrics are specific to this sample and may differ when original data is used.

This notebook demonstrates purchase-history-based recommendations. Browsing behavior is not included in the available sample tables.
___

# Setup


Install the project dependencies from `requirements.txt` before running this notebook.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import timedelta


In [ ]:
import sys
from pathlib import Path

repo_root = Path.cwd()
if not (repo_root / "data" / "sample").is_dir():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))

from src.data import clean_tables, load_raw_tables

data_dir = repo_root / "data" / "sample"
customers_df, products_df, sales_df = load_raw_tables(data_dir)
customers_df, products_df, sales_df = clean_tables(
    customers_df, products_df, sales_df
)


# 1. Data Acquisition

Load the source tables and perform initial data cleaning.


In [ ]:
customers_df.head()

In [ ]:
products_df.head()

In [ ]:
sales_df.head()

# 2. Exploratory Data Analysis

Explore the main data volumes.


### 2.1 Products


In [ ]:
products_df['LEVEL_1'].value_counts(normalize=True).sort_values(ascending=True).plot(kind='barh')

In [ ]:
product_counts=products_df.groupby(['LEVEL_1', 'LEVEL_2', 'LEVEL_3'])['PRODUCT_ID'].count().reset_index()

In [ ]:
import plotly.express as px
fig = px.sunburst(product_counts, path=['LEVEL_1', 'LEVEL_2', 'LEVEL_3'],values='PRODUCT_ID')
fig.show()

### 2.2 Sales


In [ ]:
sales_df['SALE_CATEGORY'].value_counts(normalize=True).sort_values(ascending=True).plot(kind='barh')

In [ ]:
sales_df["ORDER_KEY"] = (
    sales_df["BRANCH"].astype("string")
    + "_"
    + sales_df["SALE_ID"].astype("string").str.zfill(8)
)


In [ ]:
orders = sales_df.groupby(['ORDER_KEY','BRANCH']).agg({'PRODUCT_ID':'nunique','QUANTITY':'sum'}).reset_index()
orders.head(10)

In [ ]:
box_order = list(orders.groupby('BRANCH')['PRODUCT_ID'].mean().sort_values(ascending=False).index)

plt.figure(figsize=(16,8))
plt.xticks(rotation=65)
plt.title('Average Basket Size', fontsize=20)

sns.boxplot(data=orders, x='BRANCH', y='PRODUCT_ID', order=box_order)
plt.show()

# 3. Models


### 3.1 Collaborative Filtering


Collaborative filtering uses relationships between products and customer interests. It learns from purchase interactions to find products that may be relevant to customers with similar histories.


In [ ]:
top_similar_items = 20
branch_id = 101
days_window = 60
recent_customer_days = 5
min_purchase_count = 5
recommendation_count = 20

def get_recent_branch_sales(branch_id, lookback_days):
    branch_sales = sales_df.loc[sales_df["BRANCH"].eq(branch_id)].copy()
    if branch_sales.empty:
        raise ValueError(f"No sales found for branch {branch_id}.")
    latest_date = branch_sales["SALE_DATE"].max()
    cutoff_date = latest_date - pd.Timedelta(days=lookback_days)
    return branch_sales.loc[branch_sales["SALE_DATE"].ge(cutoff_date)].copy(), latest_date


To keep memory usage manageable, this example uses recent data from branch 101 (e-commerce).


In [ ]:
# Use the branch-specific latest date to define the interaction window.
recent_sales, latest_sale_date = get_recent_branch_sales(branch_id, days_window)
purchase_history = recent_sales[
    ["SALE_DATE", "ORDER_KEY", "PRODUCT_ID", "CUSTOMER_ID"]
].dropna(subset=["ORDER_KEY", "PRODUCT_ID", "CUSTOMER_ID"]).copy()
purchase_history.head()


In [ ]:
# Keep customers with enough distinct orders to build stable similarities.
frequent_customers = (
    purchase_history.groupby("CUSTOMER_ID")["ORDER_KEY"]
    .nunique().loc[lambda counts: counts > min_purchase_count]
)
filtered_purchase_history = purchase_history.loc[
    purchase_history["CUSTOMER_ID"].isin(frequent_customers.index)
].copy()


The model uses up to 60 days of purchase history for this branch to recommend products to recent customers. Recent customers are included even if they do not meet the minimum purchase count used to build the similarity matrix.


In [ ]:
# Retain customers active in the latest five days, including short-history customers.
recent_cutoff = latest_sale_date - pd.Timedelta(days=recent_customer_days)
recent_purchases = purchase_history.loc[
    purchase_history["SALE_DATE"].ge(recent_cutoff)
].copy()
filtered_purchase_history = pd.concat(
    [filtered_purchase_history, recent_purchases], ignore_index=True
).drop_duplicates(["ORDER_KEY", "CUSTOMER_ID", "PRODUCT_ID"])


In [ ]:
from scipy import sparse
from sklearn.metrics.pairwise import cosine_similarity
def calculate_similarity(item_data):
    """Calculate cosine similarity for each item 
    and return a DataFrame containing the item similarity matrix
    """
    sparse_item_data = sparse.csr_matrix(item_data)
    similarities = cosine_similarity(sparse_item_data.transpose())
    similarity_matrix = pd.DataFrame(data=similarities, index=item_data.columns, columns=item_data.columns)
    return similarity_matrix

def train_model(customer_product_data, normalize=True):
    """ Train the model
    """
    if normalize:
        # Normalize each customer vector so values range from 0 to 1
        magnitude = np.sqrt(np.square(customer_product_data).sum(axis=1)) # Square root of the sum of squared purchase counts
        customer_product_data = customer_product_data.div(magnitude, axis='index') # Divide purchase counts by their magnitude

    # Build the similarity matrix
    item_similarity_matrix = calculate_similarity(customer_product_data)

    return item_similarity_matrix

In [ ]:
# Count distinct orders containing each customer-product pair.
customer_product_interactions = (
    filtered_purchase_history.groupby(["CUSTOMER_ID", "PRODUCT_ID"])["ORDER_KEY"]
    .nunique().rename("PURCHASE_COUNT").reset_index()
)
customer_product_interactions.head()


In [ ]:
customer_product_matrix = customer_product_interactions.pivot_table(
    index="CUSTOMER_ID", columns="PRODUCT_ID", values="PURCHASE_COUNT",
    fill_value=0, aggfunc="sum"
)
customer_product_matrix.head()


In [ ]:
# Train the model to generate product similarities
item_similarity_matrix = train_model(customer_product_matrix, normalize=True)
item_similarity_matrix.head()

In [ ]:
# Store each product’s nearest neighbors, excluding the product itself.
item_neighbors = pd.Series(
    {
        product_id: item_similarity_matrix[product_id]
        .drop(index=product_id, errors="ignore")
        .nlargest(top_similar_items).index.tolist()
        for product_id in item_similarity_matrix.columns
    }
)
item_neighbors.head()


In [ ]:
recent_customer_ids = recent_purchases["CUSTOMER_ID"].dropna().unique()
recent_customer_ids


In [ ]:
# Score unseen candidates against the products each customer actually bought.
recommendation_rows = []
users_without_history = []
users_without_candidates = []

magnitude = np.sqrt(np.square(customer_product_matrix).sum(axis=1))
item_data = customer_product_matrix.div(magnitude.replace(0, np.nan), axis="index").fillna(0)

for user in recent_customer_ids:
    if user not in item_data.index:
        users_without_history.append(user)
        continue

    purchased_products = item_data.loc[user].loc[lambda values: values.gt(0)].index.tolist()
    neighbor_lists = [item_neighbors.get(product, []) for product in purchased_products]
    candidate_products = list(dict.fromkeys(
        candidate for neighbors in neighbor_lists for candidate in neighbors
        if candidate not in purchased_products
    ))
    if not candidate_products:
        users_without_candidates.append(user)
        continue

    similarities = item_similarity_matrix.loc[candidate_products, purchased_products]
    customer_weights = item_data.loc[user, purchased_products]
    denominator = similarities.sum(axis=1).replace(0, np.nan)
    scores = similarities.dot(customer_weights).div(denominator).dropna()
    top_scores = scores.nlargest(recommendation_count)
    recommendation_rows.extend(
        {"CUSTOMER_ID": user, "PRODUCT_ID": product_id, "SCORE": score}
        for product_id, score in top_scores.items()
    )

recommendations = pd.DataFrame(
    recommendation_rows, columns=["CUSTOMER_ID", "PRODUCT_ID", "SCORE"]
)
recommendations["CUSTOMER_ID"] = recommendations["CUSTOMER_ID"].astype("Int64")
recommendations["PRODUCT_ID"] = recommendations["PRODUCT_ID"].astype("Int64")
recommendations["SCORE"] = pd.to_numeric(recommendations["SCORE"], errors="coerce")
recommendations.head(50)


In [ ]:
recommendations = recommendations.merge(
    customers_df, on="CUSTOMER_ID", how="left", validate="many_to_one"
).merge(
    products_df, on="PRODUCT_ID", how="left", validate="many_to_one"
)
recommendations.sort_values(
    ["CUSTOMER_ID", "SCORE"], ascending=[True, False]
).head(50)


In [ ]:
recommendations[[
    "CUSTOMER_ID", "PRODUCT_ID", "SCORE",
    "LEVEL_1", "LEVEL_2", "LEVEL_3", "LEVEL_4",
]].head(20)


The resulting list contains recommendations for recent customers based on their purchase histories. It could support a CRM campaign or an online storefront. These examples are generated from sample data and have not been validated as production recommendations.


### 3.2 Basket Analysis


In [ ]:
import shutil
from IPython.display import display
from mlxtend.frequent_patterns import apriori, association_rules
from graphviz import Digraph
from matplotlib import cm

def product_label(product_id):
    match = products_df.loc[products_df["PRODUCT_ID"].eq(product_id)]
    if match.empty:
        return str(product_id)
    row = match.iloc[0]
    parts = [str(row[column]) for column in ["LEVEL_1", "LEVEL_2", "LEVEL_3", "LEVEL_4"]
             if pd.notna(row[column]) and str(row[column]).strip()]
    return " / ".join(parts) if parts else str(product_id)


This section explores relationships between products purchased together.

First, frequent itemsets are identified: groups of products that occur together in transactions. The key measures used to assess association rules are summarized below.

---
**Support**: The frequency with which an itemset appears in the dataset.

Association rules can be derived from frequent itemsets and assessed using measures such as confidence and lift.

---
**Confidence**

Of all transactions containing **A**, how many also contain **B**?

$Confidence(A \Rightarrow B) = \frac{Support(A, B)}{Support(A)}$

---
**Lift**

How much more often does **B** occur with **A** than expected from the support of **B** alone?

$Lift(A \Rightarrow B) = \frac{Confidence(A, B)}{Support(B)}$

- Lift = 1: no association beyond the baseline frequency.
- Lift > 1: positive association.
- Lift < 1: negative association.


As with collaborative filtering, this example uses a smaller dataset to manage memory usage. It uses branch 101 (e-commerce).


In [ ]:
# Build one basket per order in the branch-specific 60-day window.
basket_sales, basket_latest_date = get_recent_branch_sales(branch_id, days_window)
basket_sales = basket_sales.dropna(subset=["ORDER_KEY", "PRODUCT_ID"])
basket_matrix = (
    basket_sales.groupby(["ORDER_KEY", "PRODUCT_ID"]).size()
    .unstack(fill_value=0).gt(0)
)
basket_matrix.head()


In [ ]:
# Apriori expects a boolean transaction-by-product matrix.
binary_basket_matrix = basket_matrix.astype(bool)
binary_basket_matrix.head()


In [ ]:
minimum_support = 0.002  # 0.2% of distinct orders
frequent_itemsets = apriori(
    binary_basket_matrix, min_support=minimum_support, use_colnames=True
)
frequent_itemsets.sort_values("support", ascending=False)


The minimum support is set to 0.2% of orders and the minimum confidence to 10%. These exploratory thresholds should be reassessed with the intended business data.


In [ ]:
minimum_confidence = 0.10
if (not frequent_itemsets.empty) and frequent_itemsets["itemsets"].map(len).max() >= 2:
    association_rules_df = association_rules(
        frequent_itemsets, metric="confidence", min_threshold=minimum_confidence,
        num_itemsets=len(binary_basket_matrix)
    )
else:
    association_rules_df = pd.DataFrame(
        columns=["antecedents", "consequents", "support", "confidence", "lift"]
    )

rule_preview = association_rules_df.sort_values(
    ["lift", "confidence"], ascending=False
).head(20).copy()
if not rule_preview.empty:
    rule_preview["antecedents"] = rule_preview["antecedents"].map(
        lambda items: ", ".join(product_label(item) for item in sorted(items, key=str))
    )
    rule_preview["consequents"] = rule_preview["consequents"].map(
        lambda items: ", ".join(product_label(item) for item in sorted(items, key=str))
    )
rule_preview


Frequent itemset support is calculated over distinct orders. The exact values and rules depend on the sample and may change with the intended business data.


In [ ]:
from matplotlib import cm

def normalize_value(value, min_value, max_value):
    if min_value == max_value:
        return 0.5
    return (value - min_value) / (max_value - min_value)

def plot_association_graph(rules, image_size, color_schema, n_top, order_by,
                           rule_color="lift", rule_width="support"):
    graph = Digraph(strict=True, format="png")
    graph.attr(size=image_size, splines="line")
    if rules.empty:
        return graph

    cmap = cm.get_cmap(color_schema, 100)
    selected = rules.sort_values(order_by, ascending=False).head(n_top).copy()
    min_color, max_color = selected[rule_color].min(), selected[rule_color].max()
    min_width, max_width = selected[rule_width].min(), selected[rule_width].max()
    for _, row in selected.iterrows():
        antecedents = [product_label(item) for item in sorted(row["antecedents"], key=str)]
        consequents = [product_label(item) for item in sorted(row["consequents"], key=str)]
        rule_node = " → ".join([", ".join(antecedents), ", ".join(consequents)])
        color_value = normalize_value(row[rule_color], min_color, max_color)
        width_value = normalize_value(row[rule_width], min_width, max_width) + 0.2
        graph.node(rule_node, "", style="filled",
                   color=cm.colors.to_hex(cmap(color_value)), shape="circle",
                   width=str(width_value))
        for item in antecedents:
            graph.node(item, item, shape="plaintext")
            graph.edge(item, rule_node, color="grey")
        for item in consequents:
            graph.node(item, item, shape="plaintext")
            graph.edge(rule_node, item, color="grey")
    return graph


In [ ]:
if association_rules_df.empty:
    print("No association rules met the selected support and confidence thresholds.")
elif shutil.which("dot") is None:
    print("Graphviz executable `dot` is unavailable; showing the readable rule table instead.")
    display(rule_preview)
else:
    display(plot_association_graph(
        association_rules_df, "10", "Blues", 25, order_by="lift",
        rule_color="confidence", rule_width="support"
    ))


In [ ]:
if association_rules_df.empty:
    print("No example rule is available for the current thresholds.")
else:
    top_rule = association_rules_df.sort_values(
        ["lift", "confidence"], ascending=False
    ).head(1).copy()
    top_rule["antecedents"] = top_rule["antecedents"].map(
        lambda items: ", ".join(product_label(item) for item in sorted(items, key=str))
    )
    top_rule["consequents"] = top_rule["consequents"].map(
        lambda items: ", ".join(product_label(item) for item in sorted(items, key=str))
    )
    display(top_rule)


Frequent itemsets and association rules are calculated over orders in this sample. Readable product names are included in the rule previews and graph. Associations show co-occurrence, not causation; results may differ with the original business data.
